In [ ]:
import sys
import os
import time
import math
import numpy as np
import yaml
sys.path.append(os.path.abspath('..'))
from drone_env.drone_sim import RoomDroneEnv
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import pybullet as p

STAGE_TO_WATCH = 0

# RENDER_STRIDE: physics steps per GUI frame. Set to 10 for ~real-time, 1 for slow-motion.
RENDER_STRIDE = 10

print(f"Frozen Model Viewer initialized. Loading Stage {STAGE_TO_WATCH}...")
print(f"RENDER_STRIDE={RENDER_STRIDE}  (1 = slow-motion, 10 = ~real-time)")

config_path = os.path.abspath(os.path.join('..', 'configs', 'teacher_ppo.yaml'))
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

stage_key = f"stage_{STAGE_TO_WATCH}"
stage_config = config['stages'][stage_key]

NUM_OBS = stage_config['num_obstacles']
RAND_OBS = stage_config['randomize_obstacles']
RAND_COINS = stage_config['randomize_coins']
HOVER_ONLY = stage_config.get('hover_only', False)
NUM_FIXED_COINS = stage_config.get('num_fixed_coins', 4)
FIXED_SPAWN = stage_config.get('fixed_spawn', False)
RUN_NAME = stage_config['run_name']
reward_weights = config['hover_rewards'] if HOVER_ONLY else config['nav_rewards']

env_raw = RoomDroneEnv(gui=True, num_obstacles=NUM_OBS, randomize_obstacles=RAND_OBS,
                       randomize_coins=RAND_COINS, reward_weights=reward_weights,
                       hover_only=HOVER_ONLY, num_fixed_coins=NUM_FIXED_COINS,
                       fixed_spawn=FIXED_SPAWN)
env_vec = DummyVecEnv([lambda e=env_raw: e])

model_path = os.path.abspath(os.path.join('..', 'models', RUN_NAME, 'best_model'))
vecnorm_path = f"{model_path}_vecnormalize.pkl"
print(f"Loading frozen brain from: {model_path}.zip")

try:
    env = VecNormalize.load(vecnorm_path, env_vec)
    env.training = False
    env.norm_reward = False
    model = PPO.load(model_path, env=env)
except Exception as e:
    print(f"ERROR: Model or normalization stats not found!\nDetails: {e}")
    raise

obs = env.reset()
p.resetDebugVisualizerCamera(cameraDistance=6.0, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0, 0, 1])

def _draw_target_marker(target_pos):
    vis = p.createVisualShape(p.GEOM_SPHERE, radius=0.15, rgbaColor=[1, 1, 0, 0.8])
    p.createMultiBody(baseMass=0, baseVisualShapeIndex=vis, basePosition=list(target_pos))

def _draw_spawn_marker(pos):
    vis = p.createVisualShape(p.GEOM_SPHERE, radius=0.1, rgbaColor=[0.2, 1.0, 0.1, 1.0])
    p.createMultiBody(baseMass=0, baseVisualShapeIndex=vis, basePosition=list(pos))

if HOVER_ONLY:
    _draw_target_marker(env_raw.hover_target)
spawn_pos_init, _ = p.getBasePositionAndOrientation(env_raw.drone_id)
_draw_spawn_marker(list(spawn_pos_init))

print(f"[{RUN_NAME}] Frozen simulation started. Model will NOT update between episodes.")

_shaft = None
_head_l = None
_head_r = None
_debug_text = None
_prev_trail_pos = None
episode_reward = 0.0
ep_steps = 0
ep_start_time = time.time()
step_reward = 0.0
episode_done = False

def _draw_arrow(base, nose_dir, shaft_id, head_l_id, head_r_id):
    RED = [1, 0, 0]
    W = 2
    tip = np.array(base) + 0.5 * nose_dir
    up = np.array([0, 0, 1])
    perp = np.cross(nose_dir, up)
    mag = np.linalg.norm(perp)
    perp = perp / mag if mag > 0.01 else np.array([1, 0, 0])
    head_base = tip - 0.12 * nose_dir
    left_pt  = (head_base + 0.07 * perp).tolist()
    right_pt = (head_base - 0.07 * perp).tolist()
    tip = tip.tolist()
    base = list(base)
    shaft_id  = p.addUserDebugLine(base, tip,      RED, W, lifeTime=0, **({'replaceItemUniqueId': shaft_id}  if shaft_id  is not None else {}))
    head_l_id = p.addUserDebugLine(tip,  left_pt,  RED, W, lifeTime=0, **({'replaceItemUniqueId': head_l_id} if head_l_id is not None else {}))
    head_r_id = p.addUserDebugLine(tip,  right_pt, RED, W, lifeTime=0, **({'replaceItemUniqueId': head_r_id} if head_r_id is not None else {}))
    return shaft_id, head_l_id, head_r_id

while True:
    episode_done = False
    for _ in range(RENDER_STRIDE):
        action, _states = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = env.step(action)
        step_reward = rewards[0]
        episode_reward += step_reward
        ep_steps += 1
        if dones[0]:
            episode_done = True
            break

    if episode_done:
        real_seconds = time.time() - ep_start_time
        sim_seconds  = ep_steps / 240.0
        effective_hz = ep_steps / real_seconds if real_seconds > 0 else 0
        print(f"--- Episode End ---")
        print(f"  Steps:        {ep_steps}")
        print(f"  Sim time:     {sim_seconds:.2f}s  (steps / 240Hz)")
        print(f"  Real time:    {real_seconds:.2f}s  (wall clock)")
        print(f"  Effective Hz: {effective_hz:.1f}  (240 = true real-time)")
        print(f"  Ep Reward:    {episode_reward:.1f}")
        episode_reward = 0.0
        ep_steps = 0
        step_reward = 0.0
        _shaft = _head_l = _head_r = _debug_text = None
        _prev_trail_pos = None
        time.sleep(0.5)
        if HOVER_ONLY:
            _draw_target_marker(env_raw.hover_target)
        new_spawn, _ = p.getBasePositionAndOrientation(env_raw.drone_id)
        _draw_spawn_marker(list(new_spawn))
        # Frozen viewer: model does NOT reload between episodes
        ep_start_time = time.time()

    drone_pos, ori = p.getBasePositionAndOrientation(env_raw.drone_id)
    euler = p.getEulerFromQuaternion(ori)
    rot = np.array(p.getMatrixFromQuaternion(ori)).reshape(3, 3)
    nose_dir = rot @ np.array([0, 1, 0])

    _shaft, _head_l, _head_r = _draw_arrow(drone_pos, nose_dir, _shaft, _head_l, _head_r)

    if ep_steps % max(12, RENDER_STRIDE) == 0:
        if _prev_trail_pos is not None:
            p.addUserDebugLine(_prev_trail_pos, list(drone_pos), [0.2, 1.0, 0.1], 1, lifeTime=0)
        _prev_trail_pos = list(drone_pos)

    if HOVER_ONLY:
        dist = np.linalg.norm(np.array(drone_pos) - np.array(env_raw.hover_target))
    elif env_raw.gold_data:
        dist = min(math.sqrt(sum((a - b) ** 2 for a, b in zip(drone_pos, gd["pos"]))) for gd in env_raw.gold_data)
    else:
        dist = 0.0

    tilt_deg = math.degrees(math.sqrt(euler[0] ** 2 + euler[1] ** 2))
    ep_real_seconds = time.time() - ep_start_time
    text = (f"Alt:{drone_pos[2]:.2f}m  Dist:{dist:.2f}m  Tilt:{tilt_deg:.1f}deg"
            f"  T:{ep_real_seconds:.0f}s  r:{step_reward:.3f}  Ep R:{episode_reward:.1f}"
            f"  x{RENDER_STRIDE}")
    text_pos = [drone_pos[0], drone_pos[1], drone_pos[2] + 1.2]
    if _debug_text is None:
        _debug_text = p.addUserDebugText(text, text_pos, textColorRGB=[1, 0, 0], textSize=1.2, lifeTime=0)
    else:
        _debug_text = p.addUserDebugText(text, text_pos, textColorRGB=[1, 0, 0], textSize=1.2, lifeTime=0,
                                         replaceItemUniqueId=_debug_text)